# Ovarian Ultrasound Classification — 4 Model Benchmark
**Models:** EfficientNet-B4 (Baseline) · EfficientNet-B4 + CBAM · MobileNetV3-Large · ViT-B/16  
**Dataset:** Ovarian Ultrasound (5 classes, ~6 879 images)  
**Task:** Multi-class image classification for clinical decision support

In [ ]:
!pip install scikit-learn seaborn matplotlib --quiet

In [ ]:
import os
import copy
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, SubsetRandomSampler
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from torchvision.models import vit_b_16, ViT_B_16_Weights

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
base_dir = '/kaggle/input/datasets/ucimachinelearning/ovarian-ultrasound-image-dataset/Ovarian_US'
batch_size = 32

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=base_dir, transform=train_transforms)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"Classes: {class_names}")

dataset_size = len(full_dataset)
indices = list(range(dataset_size))
np.random.shuffle(indices)

train_split, val_split = int(0.70 * dataset_size), int(0.85 * dataset_size)
train_idx = indices[:train_split]
val_idx   = indices[train_split:val_split]
test_idx  = indices[val_split:]

train_sampler = SubsetRandomSampler(train_idx)
val_sampler   = SubsetRandomSampler(val_idx)
test_sampler  = SubsetRandomSampler(test_idx)

train_loader = DataLoader(full_dataset, batch_size=batch_size, sampler=train_sampler, num_workers=2)
clean_dataset = datasets.ImageFolder(root=base_dir, transform=test_transforms)
val_loader  = DataLoader(clean_dataset, batch_size=batch_size, sampler=val_sampler,  num_workers=2)
test_loader = DataLoader(clean_dataset, batch_size=batch_size, sampler=test_sampler, num_workers=2)

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

In [ ]:
# ─── CBAM Attention Modules ───────────────────────────────────────────────────

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x_cat))


class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x


# ─── Model 1: EfficientNet-B4 Baseline ────────────────────────────────────────

class EfficientNetBaseline(nn.Module):
    def __init__(self, num_classes=5):
        super(EfficientNetBaseline, self).__init__()
        self.backbone = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
        for param in self.backbone.features.parameters():
            param.requires_grad = False
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)


# ─── Model 2: EfficientNet-B4 + CBAM ──────────────────────────────────────────

class EfficientNetCBAM(nn.Module):
    def __init__(self, num_classes=5):
        super(EfficientNetCBAM, self).__init__()
        self.backbone = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
        for param in self.backbone.features.parameters():
            param.requires_grad = False
        in_features = self.backbone.classifier[1].in_features
        self.features    = self.backbone.features
        self.cbam        = CBAM(in_planes=in_features)
        self.pool        = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.cbam(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


# ─── Model 3: MobileNetV3-Large ───────────────────────────────────────────────

class MobileNetV3Model(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        for param in self.backbone.features.parameters():
            param.requires_grad = False
        in_feats = self.backbone.classifier[3].in_features
        self.backbone.classifier[3] = nn.Linear(in_feats, num_classes)

    def forward(self, x):
        return self.backbone(x)


# ─── Model 4: ViT-B/16 ────────────────────────────────────────────────────────

class ViTModel(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        for param in self.backbone.parameters():
            param.requires_grad = False
        self.backbone.heads.head = nn.Linear(self.backbone.heads.head.in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)


# Instantiate all four models
model_baseline = EfficientNetBaseline(num_classes=num_classes).to(device)
model_cbam     = EfficientNetCBAM(num_classes=num_classes).to(device)
model_mobilenet= MobileNetV3Model(num_classes=num_classes).to(device)
model_vit      = ViTModel(num_classes=num_classes).to(device)

print("All 4 models instantiated successfully.")

In [ ]:
EPOCHS = 15

def train_model(model, name):
    print(f"\n--- Training {name} ---")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    history  = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

    for epoch in range(EPOCHS):
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss, running_corrects = 0.0, 0

            for inputs, labels in dataloader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss      += loss.item() * inputs.size(0)
                running_corrects  += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(dataloader.sampler)
            epoch_acc  = running_corrects.double() / len(dataloader.sampler)

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            if phase == 'val':
                print(f"Epoch {epoch+1}/{EPOCHS} | Val Acc: {epoch_acc:.4f} | Val Loss: {epoch_loss:.4f}")
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_wts)
    print(f"-> Best {name} Val Acc: {best_acc:.4f}")
    return history, best_acc.item()


def evaluate_on_test(model, name):
    model.eval()
    running_corrects = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs   = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    test_acc = (running_corrects.double() / len(test_loader.sampler)).item()
    return test_acc, np.array(all_labels), np.array(all_preds), np.array(all_probs)

print("Training and evaluation functions ready.")

In [ ]:
# ─── Train Model 1 & 2: EfficientNet-B4 Baseline and +CBAM ───────────────────

history_base, val_acc_base = train_model(model_baseline, "EfficientNet-B4 (Baseline)")
history_cbam, val_acc_cbam = train_model(model_cbam,     "EfficientNet-B4 + CBAM")

test_acc_base, _, _, _                           = evaluate_on_test(model_baseline, "Baseline")
test_acc_cbam, labels_cbam, preds_cbam, probs_cbam = evaluate_on_test(model_cbam,  "CBAM")

print("="*40)
print("ACCURACY COMPARISON — EfficientNet")
print("="*40)
print(f"Baseline  | Val: {val_acc_base:.4f} | Test: {test_acc_base:.4f}")
print(f"+ CBAM    | Val: {val_acc_cbam:.4f} | Test: {test_acc_cbam:.4f}")
print("="*40)

# Val accuracy curve
plt.figure(figsize=(10, 5))
plt.plot(history_base['val_acc'], label='Val Acc (Baseline)', linestyle='--', color='red')
plt.plot(history_cbam['val_acc'], label='Val Acc (+ CBAM)',   linewidth=2.5, color='blue')
plt.title('Validation Accuracy: EfficientNet-B4 Baseline vs CBAM')
plt.xlabel('Epochs'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

# Multiclass ROC (CBAM model)
y_bin = label_binarize(labels_cbam, classes=[0,1,2,3,4])
plt.figure(figsize=(10, 8))
for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], probs_cbam[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'ROC {class_names[i]} (AUC = {roc_auc:.3f})')
plt.plot([0,1],[0,1], color='black', lw=2, linestyle='--')
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — EfficientNet-B4 + CBAM')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

# Confusion matrix (CBAM)
cm = confusion_matrix(labels_cbam, preds_cbam)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Ground Truth')
plt.title('Confusion Matrix — EfficientNet-B4 + CBAM')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Train Model 3 & 4: MobileNetV3-Large and ViT-B/16 ───────────────────────

history_mob, val_acc_mob = train_model(model_mobilenet, "MobileNetV3-Large")
history_vit, val_acc_vit = train_model(model_vit,       "ViT-B/16")

test_acc_mob, _, _, _ = evaluate_on_test(model_mobilenet, "MobileNetV3")
test_acc_vit, _, _, _ = evaluate_on_test(model_vit,       "ViT-B/16")

print("="*40)
print("ACCURACY COMPARISON — Additional Models")
print("="*40)
print(f"MobileNetV3-Large | Val: {val_acc_mob:.4f} | Test: {test_acc_mob:.4f}")
print(f"ViT-B/16          | Val: {val_acc_vit:.4f} | Test: {test_acc_vit:.4f}")
print("="*40)

In [ ]:
# ─── Final Leaderboard & Comparison Chart ────────────────────────────────────

model_names = ['EfficientNet-B4\n(Baseline)', 'EfficientNet-B4\n+CBAM',
               'MobileNetV3\nLarge', 'ViT-B/16']
test_accs   = [test_acc_base, test_acc_cbam, test_acc_mob, test_acc_vit]
colors      = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']

plt.figure(figsize=(10, 6))
bars = plt.bar(model_names, [a*100 for a in test_accs],
               color=colors, edgecolor='black', linewidth=1.2)
for bar, acc in zip(bars, test_accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{acc*100:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
plt.title('Test Accuracy Comparison — All 4 Models', fontsize=14, fontweight='bold')
plt.ylabel('Test Accuracy (%)'); plt.ylim(0, 105)
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

print("\n=== Final Leaderboard ===")
for name, acc in zip(model_names, test_accs):
    print(f"{name.replace(chr(10),' '):<35} Test Acc: {acc*100:.2f}%")